# Vega reprojection on a reduced grid

Project a fine-grid swaption/cap-floor vega matrix onto a coarser
(expiry × tenor) ladder, with full per-bucket diagnostics for reliability.

**Methods covered**
- Bilinear (tent-basis) weighting — transparent baseline, exact total preservation
- Constrained least-squares projection — KKT solve, equality constraints
- Diagnostics: residuals, leverage, SVD subspace, LOO error, bootstrap uncertainty

**Outputs**
- Heatmaps of original / projected / residual / leverage / bootstrap-std
- Per-bucket reliability table
- Scalar quality summary

Replace the synthetic data block with your real vega matrix to use in production.


## 1. Setup and synthetic vega book

We build a synthetic vega matrix that mimics a real swaption book: a few
clusters of large positions (long and short), plus low-amplitude noise
across the surface. Replace `M_orig` with your actual matrix to use this
notebook on a real book.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

rng = np.random.default_rng(42)

# Original (fine) grid
E_orig = np.array([1/12, 3/12, 6/12, 9/12, 1, 1.5, 2, 3, 5, 7, 10, 15, 20, 30])
T_orig = np.array([1, 2, 3, 5, 7, 10, 15, 20, 25, 30])
nE, nT = len(E_orig), len(T_orig)

E_labels = ['1M','3M','6M','9M','1Y','18M','2Y','3Y','5Y','7Y','10Y','15Y','20Y','30Y']
T_labels = ['1Y','2Y','3Y','5Y','7Y','10Y','15Y','20Y','25Y','30Y']

def synth_vega(E, T, rng):
    EE, TT = np.meshgrid(np.log(E), np.log(T), indexing='ij')
    M = np.zeros_like(EE)
    # (log_expiry_centre, log_tenor_centre, amplitude, sigma_E, sigma_T)
    centers = [
        (np.log(2),   np.log(10), +1500.0, 0.40, 0.50),  # long 2y x 10y
        (np.log(5),   np.log(5),   -800.0, 0.30, 0.40),  # short 5y x 5y
        (np.log(0.5), np.log(2),   +400.0, 0.50, 0.60),  # long 6m x 2y
        (np.log(10),  np.log(30),  +600.0, 0.50, 0.50),  # long 10y x 30y
    ]
    for e0, t0, amp, se, st in centers:
        M += amp * np.exp(-0.5 * (((EE - e0)/se)**2 + ((TT - t0)/st)**2))
    M += rng.normal(0.0, 30.0, size=M.shape)
    return M

M_orig = synth_vega(E_orig, T_orig, rng)
v_orig = M_orig.flatten()

print(f"Original grid: {nE} expiries x {nT} tenors = {nE*nT} buckets")
print(f"Total vega: {v_orig.sum():,.2f}")
print(f"Min / max bucket: {v_orig.min():,.2f} / {v_orig.max():,.2f}")


In [ ]:
def heatmap(ax, M, row_labels, col_labels, title, cmap='RdBu_r', diverging=True, fmt='{:.0f}'):
    if diverging:
        vmax = max(abs(M.min()), abs(M.max()))
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        im = ax.imshow(M, aspect='auto', cmap=cmap, norm=norm)
    else:
        im = ax.imshow(M, aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right')
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_xlabel('Tenor'); ax.set_ylabel('Expiry')
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, fmt.format(M[i, j]), ha='center', va='center',
                    color='black', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return im

fig, ax = plt.subplots(figsize=(11, 7))
heatmap(ax, M_orig, E_labels, T_labels, 'Original vega (synthetic book)')
plt.tight_layout(); plt.show()


## 2. Reduced grid

Define the desk-standard ladder you want to project onto. Pick something
sparse enough to be reportable but dense enough to capture the dominant
exposure shapes.


In [ ]:
E_red = np.array([3/12, 1, 2, 5, 10, 30])
T_red = np.array([2, 5, 10, 30])
mE, mT = len(E_red), len(T_red)
m, n = mE * mT, nE * nT

E_red_labels = ['3M','1Y','2Y','5Y','10Y','30Y']
T_red_labels = ['2Y','5Y','10Y','30Y']

print(f"Reduced grid: {mE} expiries x {mT} tenors = {m} buckets")
print(f"Compression ratio: {n}/{m} = {n/m:.1f}x")


## 3. Bilinear (tent) basis matrix

Build `B` of shape `(n, m)`. Each row corresponds to an original grid point
and has up to four non-zero entries: the bilinear weights distributing that
point's vega to the four surrounding reduced nodes in **log-space**.

By construction every row sums to 1, so `B.T @ v_orig` preserves total vega
exactly. The same matrix `B` doubles as the design matrix for the
least-squares projection in step 5.


In [ ]:
def bilinear_basis(E_orig, T_orig, E_red, T_red):
    """Build (n, m) tent-basis matrix in log(time) space, with edge clamping."""
    nE, nT = len(E_orig), len(T_orig)
    mE, mT = len(E_red), len(T_red)
    B = np.zeros((nE * nT, mE * mT))
    logE_red, logT_red = np.log(E_red), np.log(T_red)

    for i, e in enumerate(E_orig):
        le = np.log(e)
        if le <= logE_red[0]:
            ie_lo, ie_hi, alpha = 0, 0, 0.0
        elif le >= logE_red[-1]:
            ie_lo, ie_hi, alpha = mE - 1, mE - 1, 0.0
        else:
            ie_hi = np.searchsorted(logE_red, le)
            ie_lo = ie_hi - 1
            alpha = (le - logE_red[ie_lo]) / (logE_red[ie_hi] - logE_red[ie_lo])

        for j, t in enumerate(T_orig):
            lt = np.log(t)
            if lt <= logT_red[0]:
                it_lo, it_hi, beta = 0, 0, 0.0
            elif lt >= logT_red[-1]:
                it_lo, it_hi, beta = mT - 1, mT - 1, 0.0
            else:
                it_hi = np.searchsorted(logT_red, lt)
                it_lo = it_hi - 1
                beta = (lt - logT_red[it_lo]) / (logT_red[it_hi] - logT_red[it_lo])

            row = i * nT + j
            B[row, ie_lo*mT + it_lo] += (1-alpha)*(1-beta)
            B[row, ie_lo*mT + it_hi] += (1-alpha)*beta
            B[row, ie_hi*mT + it_lo] += alpha*(1-beta)
            B[row, ie_hi*mT + it_hi] += alpha*beta
    return B

B = bilinear_basis(E_orig, T_orig, E_red, T_red)
row_sums = B.sum(axis=1)
print(f"B shape: {B.shape}")
print(f"Row sums: min={row_sums.min():.6f}, max={row_sums.max():.6f}  (should be 1.0)")
print(f"B is sparse: {(B != 0).sum()} non-zeros out of {B.size} cells "
      f"({100*(B != 0).sum()/B.size:.1f}%)")


## 4. Bilinear projection (baseline)

The simplest projection: each original bucket distributes its vega to its
four reduced neighbours using the tent weights.

$$v_{\text{red}} = B^\top v_{\text{orig}}$$

This **exactly preserves total vega** by construction, but is not optimal
in any reconstruction sense — it's purely geometric.


In [ ]:
v_red_bilinear = B.T @ v_orig
M_red_bilinear = v_red_bilinear.reshape(mE, mT)

print(f"Bilinear total: {v_red_bilinear.sum():,.2f}")
print(f"Original total: {v_orig.sum():,.2f}")
print(f"Difference:     {v_red_bilinear.sum() - v_orig.sum():.2e}  (machine epsilon)")

fig, ax = plt.subplots(figsize=(7, 5))
heatmap(ax, M_red_bilinear, E_red_labels, T_red_labels, 'Bilinear projection')
plt.tight_layout(); plt.show()


## 5. Constrained least-squares projection

Treat the reduced vega as the unknown $v$ and the original vega as
observations $y$. The tent basis $B$ is the design matrix. Solve

$$\min_v \; \| B v - y \|^2 \quad \text{s.t.} \quad A v = b$$

with the equality constraint $\sum_k v_k = \sum_i y_i$ to preserve total
vega. The KKT system is

$$\begin{bmatrix} 2 B^\top B & A^\top \\ A & 0 \end{bmatrix} \begin{bmatrix} v \\ \lambda \end{bmatrix} = \begin{bmatrix} 2 B^\top y \\ b \end{bmatrix}$$

This is a single linear solve — no iterative optimizer, no `cvxpy`
dependency. Add per-expiry or per-tenor preservation constraints by
stacking more rows into `A` and `b`.


In [ ]:
def constrained_ls(B, y, A, b):
    """Solve  min ||Bv - y||^2  s.t.  A v = b,  via the KKT system."""
    n_v, n_c = B.shape[1], A.shape[0]
    KKT = np.zeros((n_v + n_c, n_v + n_c))
    KKT[:n_v, :n_v] = 2 * B.T @ B
    KKT[:n_v, n_v:] = A.T
    KKT[n_v:, :n_v] = A
    rhs = np.concatenate([2 * B.T @ y, b])
    sol = np.linalg.solve(KKT, rhs)
    return sol[:n_v]

# Just a total-vega preservation constraint
A_total = np.ones((1, m))
b_total = np.array([v_orig.sum()])

v_red_cls = constrained_ls(B, v_orig, A_total, b_total)
M_red_cls = v_red_cls.reshape(mE, mT)
v_recon = B @ v_red_cls
residuals = v_orig - v_recon

print(f"CLS total: {v_red_cls.sum():,.2f}  (constraint satisfied)")
print(f"Reconstruction RMSE: {np.sqrt((residuals**2).mean()):,.2f}")
print(f"Mean |residual|:     {np.abs(residuals).mean():,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
heatmap(axes[0], M_red_bilinear, E_red_labels, T_red_labels, 'Bilinear')
heatmap(axes[1], M_red_cls,      E_red_labels, T_red_labels, 'Constrained LS')
plt.tight_layout(); plt.show()


**Note on the difference between the two projections.** The bilinear
projection re-distributes vega from each fine bucket to its neighbours.
Constrained LS, by contrast, finds the reduced vega vector whose
**re-expansion** through the tent basis best matches the original — it
fights against the bias that bilinear introduces near peaks of the surface,
at the cost of being more sensitive to noise.


## 6. Diagnostics

Five views of "where is the projection trustworthy?":

1. **Residual heatmap** — which original buckets are mis-represented
2. **Leverage** — which original buckets dominate the fit
3. **LOO residuals** — out-of-sample reconstruction error per point
4. **SVD of B** — global condition + un-representable subspace
5. **Bootstrap** — uncertainty on each reduced bucket


### 6a. Residual heatmap

Reshape `v_orig - B @ v_red_cls` back to the original grid. Bright cells
are buckets the reduced grid simply cannot capture — typically peaks
between reduced nodes, or sharp local features.


In [ ]:
M_resid = residuals.reshape(nE, nT)
fig, ax = plt.subplots(figsize=(11, 7))
heatmap(ax, M_resid, E_labels, T_labels, 'Reconstruction residuals (orig - reconstructed)')
plt.tight_layout(); plt.show()

# Quantify
print(f"Max |residual|: {np.abs(residuals).max():,.2f} "
      f"at original bucket index {np.abs(residuals).argmax()}")
i_max = np.abs(residuals).argmax()
ie, it = i_max // nT, i_max % nT
print(f"  -> expiry {E_labels[ie]} x tenor {T_labels[it]}")


### 6b. Leverage (hat matrix diagonal)

For OLS the hat matrix is $H = B (B^\top B)^{-1} B^\top$ and the diagonal
$H_{ii}$ measures how much original point $i$ influences its own predicted
value. Mean leverage equals $m/n$. High-leverage points dominate the fit;
low-leverage points contribute little (their information is being
discarded).


In [ ]:
BtB_inv = np.linalg.pinv(B.T @ B)
H_diag = np.einsum('ij,jk,ik->i', B, BtB_inv, B)
M_lev = H_diag.reshape(nE, nT)

print(f"Leverage: mean={H_diag.mean():.4f} (theoretical m/n = {m/n:.4f})")
print(f"          max ={H_diag.max():.4f}")
print(f"          min ={H_diag.min():.4f}")

fig, ax = plt.subplots(figsize=(11, 7))
heatmap(ax, M_lev, E_labels, T_labels, 'Leverage H_ii', cmap='viridis',
        diverging=False, fmt='{:.2f}')
plt.tight_layout(); plt.show()


### 6c. Leave-one-out residuals

For OLS the LOO residual has a closed form:

$$r^{\text{LOO}}_i = \frac{r_i}{1 - H_{ii}}$$

This is what you'd quote as "if I dropped this trade, how badly would the
projection mis-predict it?" — a more honest reliability measure than the
in-sample residual.


In [ ]:
r_loo = residuals / np.clip(1 - H_diag, 1e-6, None)
M_loo = r_loo.reshape(nE, nT)

fig, ax = plt.subplots(figsize=(11, 7))
heatmap(ax, M_loo, E_labels, T_labels, 'Leave-one-out residuals')
plt.tight_layout(); plt.show()

print(f"LOO RMSE: {np.sqrt((r_loo**2).mean()):,.2f}  "
      f"(in-sample RMSE: {np.sqrt((residuals**2).mean()):,.2f})")


### 6d. SVD of B — global stability and un-representable subspace

The condition number $\sigma_1 / \sigma_m$ flags ill-posedness. The right
singular vectors of small singular values describe the **exposure shapes
the reduced grid cannot capture**. Project `v_orig` onto the column space
of `B` and the leftover norm is the un-representable energy.


In [ ]:
U, S, Vt = np.linalg.svd(B, full_matrices=False)
cond = S[0] / S[-1]
print(f"Singular values of B: {S.round(3)}")
print(f"Condition number: {cond:.2f}")

v_orig_in_colB = U @ (U.T @ v_orig)
unrep = v_orig - v_orig_in_colB
unrep_frac = np.linalg.norm(unrep)**2 / np.linalg.norm(v_orig)**2
print(f"Un-representable energy: {100*unrep_frac:.2f}% of total vega norm")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(S, 'o-'); axes[0].set_title('Singular values of B')
axes[0].set_xlabel('index'); axes[0].set_ylabel('sigma'); axes[0].grid(True)

M_unrep = unrep.reshape(nE, nT)
heatmap(axes[1], M_unrep, E_labels, T_labels,
        'Un-representable component of v_orig')
plt.tight_layout(); plt.show()


### 6e. Bootstrap uncertainty on reduced buckets

Resample original buckets with replacement, refit, repeat. The std of each
reduced bucket across resamples is its sampling-noise uncertainty. This is
the number to cite when reporting the projected ladder.


In [ ]:
n_boot = 500
v_red_boot = np.zeros((n_boot, m))
for k in range(n_boot):
    idx = rng.integers(0, n, size=n)
    Bk, yk = B[idx], v_orig[idx]
    try:
        v_red_boot[k] = constrained_ls(Bk, yk, A_total, b_total)
    except np.linalg.LinAlgError:
        v_red_boot[k] = v_red_cls

v_red_std = v_red_boot.std(axis=0)
M_std = v_red_std.reshape(mE, mT)

# Coefficient of variation per reduced bucket
cv = np.where(np.abs(v_red_cls) > 1e-6, v_red_std / np.abs(v_red_cls), np.nan)
M_cv = cv.reshape(mE, mT)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
heatmap(axes[0], M_std, E_red_labels, T_red_labels,
        'Bootstrap std of reduced vega', cmap='magma_r', diverging=False)
heatmap(axes[1], M_cv, E_red_labels, T_red_labels,
        'CV = std / |vega|  (lower = more reliable)',
        cmap='magma_r', diverging=False, fmt='{:.2f}')
plt.tight_layout(); plt.show()

print("Reduced buckets ranked by reliability (low CV = trustworthy):")
flat_cv = [(E_red_labels[k//mT], T_red_labels[k%mT], v_red_cls[k], v_red_std[k], cv[k])
           for k in range(m)]
flat_cv.sort(key=lambda x: (np.inf if np.isnan(x[4]) else x[4]))
for E, T, vega, std, c in flat_cv[:5]:
    print(f"  expiry {E:>4s} x tenor {T:>4s}: vega={vega:+8.1f}  std={std:6.1f}  CV={c:.2f}")
print("Worst 5:")
for E, T, vega, std, c in flat_cv[-5:]:
    print(f"  expiry {E:>4s} x tenor {T:>4s}: vega={vega:+8.1f}  std={std:6.1f}  CV={c:.2f}")


## 7. Combined dashboard

A single 2×3 panel for the desk: original surface, projected ladder
(with bootstrap std), residuals, leverage, LOO error, and CV of reduced
buckets.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

heatmap(axes[0,0], M_orig,   E_labels,     T_labels,     'Original vega')
heatmap(axes[0,1], M_red_cls, E_red_labels, T_red_labels, 'Projected vega (CLS)')
heatmap(axes[0,2], M_resid,  E_labels,     T_labels,     'Residuals (orig - recon)')
heatmap(axes[1,0], M_lev,    E_labels,     T_labels,     'Leverage H_ii',
        cmap='viridis', diverging=False, fmt='{:.2f}')
heatmap(axes[1,1], M_loo,    E_labels,     T_labels,     'LOO residuals')
heatmap(axes[1,2], M_cv,     E_red_labels, T_red_labels, 'Reduced-bucket CV',
        cmap='magma_r', diverging=False, fmt='{:.2f}')

plt.suptitle('Vega reprojection diagnostics dashboard', fontsize=14, y=1.00)
plt.tight_layout(); plt.show()


## 8. Scalar quality summary

The numbers you'd put at the top of a daily report.


In [ ]:
ss_tot = ((v_orig - v_orig.mean())**2).sum()
ss_res = (residuals**2).sum()
r2 = 1 - ss_res / ss_tot

print('=' * 60)
print('VEGA REPROJECTION SUMMARY')
print('=' * 60)
print(f'Original buckets:                {n}')
print(f'Reduced  buckets:                {m}')
print(f'Compression ratio:               {n/m:.1f}x')
print()
print(f'Total vega (original):           {v_orig.sum():>12,.2f}')
print(f'Total vega (reduced):            {v_red_cls.sum():>12,.2f}')
print(f'Total vega preservation error:   {abs(v_red_cls.sum()-v_orig.sum()):>12.2e}')
print()
print(f'Reconstruction RMSE:             {np.sqrt((residuals**2).mean()):>12,.2f}')
print(f'Reconstruction R^2:              {r2:>12.4f}')
print(f'  (negative R^2 is normal here: projection rejects bucket-level noise)')
print()
print(f'Condition number of B:           {cond:>12.2f}')
print(f'Un-representable energy:         {100*unrep_frac:>11.2f}%')
print()
print(f'Mean leverage (= m/n):           {H_diag.mean():>12.4f}')
print(f'Max  leverage:                   {H_diag.max():>12.4f}')
print()
print(f'Mean bootstrap std (red bucket): {v_red_std.mean():>12.2f}')
print(f'Max  bootstrap std (red bucket): {v_red_std.max():>12.2f}')
print('=' * 60)


## 9. Plugging in a real vega matrix

Replace section 1's `synth_vega` block with your actual matrix:

```python
# Example: load from CSV with expiries as the index, tenors as columns
import pandas as pd
df = pd.read_csv('vega_book.csv', index_col=0)
M_orig = df.values
E_orig = df.index.values.astype(float)        # expiries in years
T_orig = df.columns.values.astype(float)      # tenors in years
E_labels = [str(e) for e in df.index]
T_labels = [str(t) for t in df.columns]
nE, nT = M_orig.shape
v_orig = M_orig.flatten()
```

Then run sections 2–8 as-is. Tune the reduced grid in section 2 to match
the desk's reporting ladder.

**Optional extensions**
- Add per-expiry preservation: stack rows into `A_total` that sum the
  reduced buckets along each reduced expiry, with `b_total` containing the
  bilinear-aggregated original vega per reduced expiry.
- Add Tikhonov regularization for ill-conditioned `B`: the KKT solve
  becomes `2 (B'B + λ D'D) v + A'λ = 2 B'y` with `D` a finite-difference
  operator on the reduced grid.
- Swap the bootstrap for **block bootstrap** if your original buckets are
  spatially correlated (they usually are in a real book — adjacent buckets
  share underlying trades).
